# 02a — Strategy A: Fine-tune Pix2Pix (DStroke's hard segmentation)

Warm-start from [`huggan/pix2pix-edge2shoes`](https://huggingface.co/huggan/pix2pix-edge2shoes) and fine-tune the modified Pix2Pix from Fu et al. 2021 §3.2 on the synthetic `(painting → edge_map)` pairs produced by `01_dataset_synthesis.ipynb`.

Modifications vs vanilla Pix2Pix:

- Discriminator input is `[painting, G(x), T(G(x))]` instead of `[painting, G(x)]`, where `T(·)` is the trapped-ball region-closing operator (Eq. 3 in the paper).
- `T(·)` is implemented in `dstroke_utils.trapped_ball_merge` — it closes gaps from the blurred intensity edges so the discriminator sees a binary closed-region edge map, not a fuzzy intensity image.
- Loss = `cGAN + λ·L1(G(x), y)`, `λ = 100` (Eq. 4).

Prereqs:

- `data_notcommitted/dstroke_synth/{train,val}` populated by notebook 01.
- PyTorch with CUDA: `pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121`
- `pip install huggingface_hub safetensors`

In [ ]:
from __future__ import annotations
import os, sys, time, math
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR))
import dstroke_utils as d

REPO_ROOT = Path('C:/Users/MichelleJacobs/Repos/AIAi')
SYNTH = REPO_ROOT / 'data_notcommitted' / 'dstroke_synth'
OUT = REPO_ROOT / 'AIA - AI Tool - Fingerprinting-1' / 'outputs' / 'dstroke'
OUT.mkdir(parents=True, exist_ok=True)
CKPT = OUT / 'strategy_A_pix2pix_finetune.pt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| CUDA:', torch.cuda.is_available())
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## Dataset

Pairs live under `SYNTH/{train,val}/paintings/*.png` and `edges/*.png`. Everything is already 256×256.

In [ ]:
class SynthPairs(Dataset):
    def __init__(self, root: Path, augment: bool = False):
        self.root = Path(root)
        self.paintings = sorted((self.root / 'paintings').glob('*.png'))
        self.augment = augment

    def __len__(self) -> int:
        return len(self.paintings)

    def __getitem__(self, i: int):
        p = self.paintings[i]
        e = self.root / 'edges' / p.name
        painting = np.array(Image.open(p).convert('RGB'), dtype=np.float32) / 255.0
        edge = np.array(Image.open(e).convert('L'),   dtype=np.float32) / 255.0
        if self.augment:
            if np.random.rand() < 0.5:
                painting = painting[:, ::-1].copy()
                edge = edge[:, ::-1].copy()
        # Pix2Pix convention: inputs in [-1, 1]
        x = torch.from_numpy(painting).permute(2, 0, 1) * 2 - 1        # (3,H,W)
        y = torch.from_numpy(edge).unsqueeze(0) * 2 - 1                # (1,H,W)
        return x.float(), y.float()

train_ds = SynthPairs(SYNTH / 'train', augment=True)
val_ds   = SynthPairs(SYNTH / 'val',   augment=False)
print(f'train: {len(train_ds)}  val: {len(val_ds)}')

BATCH = 1  # paper uses bs=1
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)

## U-Net generator and PatchGAN discriminator

Standard Isola et al. 2017 Pix2Pix architecture — 8-layer U-Net at 256×256. The generator input channels can differ from `huggan/pix2pix-edge2shoes` (3→3 there vs our 3→1) so we reinitialise the final conv and keep everything upstream from the pretrained weights.

In [ ]:
def down(in_c, out_c, norm=True):
    layers = [nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False)]
    if norm:
        layers.append(nn.BatchNorm2d(out_c))
    layers.append(nn.LeakyReLU(0.2, inplace=True))
    return nn.Sequential(*layers)

def up(in_c, out_c, dropout=False):
    layers = [
        nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=False),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
    ]
    if dropout:
        layers.append(nn.Dropout(0.5))
    return nn.Sequential(*layers)

class UNetGenerator(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=64):
        super().__init__()
        self.d1 = down(in_ch, base,      norm=False)    # 128
        self.d2 = down(base,      base*2)               # 64
        self.d3 = down(base*2,    base*4)               # 32
        self.d4 = down(base*4,    base*8)               # 16
        self.d5 = down(base*8,    base*8)               # 8
        self.d6 = down(base*8,    base*8)               # 4
        self.d7 = down(base*8,    base*8)               # 2
        self.d8 = down(base*8,    base*8, norm=False)   # 1
        self.u1 = up(base*8,     base*8, dropout=True)
        self.u2 = up(base*16,    base*8, dropout=True)
        self.u3 = up(base*16,    base*8, dropout=True)
        self.u4 = up(base*16,    base*8)
        self.u5 = up(base*16,    base*4)
        self.u6 = up(base*8,     base*2)
        self.u7 = up(base*4,     base)
        self.final = nn.Sequential(
            nn.ConvTranspose2d(base*2, out_ch, 4, 2, 1),
            nn.Tanh(),
        )
    def forward(self, x):
        d1 = self.d1(x); d2 = self.d2(d1); d3 = self.d3(d2); d4 = self.d4(d3)
        d5 = self.d5(d4); d6 = self.d6(d5); d7 = self.d7(d6); d8 = self.d8(d7)
        u1 = self.u1(d8);            u1 = torch.cat([u1, d7], 1)
        u2 = self.u2(u1);            u2 = torch.cat([u2, d6], 1)
        u3 = self.u3(u2);            u3 = torch.cat([u3, d5], 1)
        u4 = self.u4(u3);            u4 = torch.cat([u4, d4], 1)
        u5 = self.u5(u4);            u5 = torch.cat([u5, d3], 1)
        u6 = self.u6(u5);            u6 = torch.cat([u6, d2], 1)
        u7 = self.u7(u6);            u7 = torch.cat([u7, d1], 1)
        return self.final(u7)

class PatchDiscriminator(nn.Module):
    """70x70 PatchGAN with 3 extra channels for the reference edge map T(x).
    Input = cat([painting (3), G(x) (1), T(G(x)) (1)]) → 5 ch.
    """
    def __init__(self, in_ch=5, base=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, base, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base, base*2, 4, 2, 1, bias=False), nn.BatchNorm2d(base*2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base*2, base*4, 4, 2, 1, bias=False), nn.BatchNorm2d(base*4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base*4, base*8, 4, 1, 1, bias=False), nn.BatchNorm2d(base*8), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base*8, 1, 4, 1, 1),
        )
    def forward(self, x):
        return self.net(x)

G = UNetGenerator(in_ch=3, out_ch=1).to(DEVICE)
D = PatchDiscriminator(in_ch=5).to(DEVICE)
print('G params:', sum(p.numel() for p in G.parameters())/1e6, 'M')
print('D params:', sum(p.numel() for p in D.parameters())/1e6, 'M')

### Warm-start generator from `huggan/pix2pix-edge2shoes`

The public checkpoint has 3→3 input/output channels (RGB shoe image from RGB edge map). We copy over every layer whose shape matches; the final conv is re-initialised because our output is 1-channel (edge map) instead of 3-channel. If loading fails for any reason, training still works but starts from random init — we log what fraction of parameters were warm-started.

In [ ]:
def warm_start(model: nn.Module, repo_id: str = 'huggan/pix2pix-edge2shoes') -> float:
    """Load state_dict from HF, keep shape-matching tensors. Returns fraction loaded."""
    try:
        from huggingface_hub import hf_hub_download
        ckpt_path = hf_hub_download(repo_id=repo_id, filename='pytorch_model.bin')
        hf_state = torch.load(ckpt_path, map_location='cpu')
    except Exception as e:
        print(f'Could not fetch {repo_id}: {e}')
        return 0.0

    own = model.state_dict()
    loaded_tensors = 0
    total_tensors = len(own)
    # The huggan repo uses slightly different key names; try a tolerant match.
    for k_own, v_own in own.items():
        # try exact match first
        if k_own in hf_state and hf_state[k_own].shape == v_own.shape:
            own[k_own] = hf_state[k_own]
            loaded_tensors += 1
            continue
        # try suffix match (e.g., 'd1.0.weight' vs 'model.down1.0.weight')
        tail = k_own.split('.', 1)[-1]
        candidates = [k for k in hf_state if k.endswith(tail) and hf_state[k].shape == v_own.shape]
        if candidates:
            own[k_own] = hf_state[candidates[0]]
            loaded_tensors += 1
    model.load_state_dict(own)
    frac = loaded_tensors / max(total_tensors, 1)
    print(f'Warm-started {loaded_tensors}/{total_tensors} tensors ({frac:.1%})')
    return frac

_ = warm_start(G)

## Training

Reference edge map `T(G(x))` is computed on CPU via `trapped_ball_merge`, then moved back to GPU. It's not differentiable — gradients only flow through `G(x)` and the `L1` term. This matches the paper's setup, where `T(·)` acts as a post-process that shapes the discriminator's view.

In [ ]:
def trapped_ball_batch(g_out: torch.Tensor) -> torch.Tensor:
    """Convert generator output (-1..1) to a closed-region binary map (-1..1)."""
    arr = ((g_out.detach().cpu().numpy() + 1) / 2).clip(0, 1)   # (B, 1, H, W)
    out = np.zeros_like(arr)
    for b in range(arr.shape[0]):
        tb = d.trapped_ball_merge(arr[b, 0])
        out[b, 0] = tb.astype(np.float32)
    t = torch.from_numpy(out * 2 - 1).to(g_out.device).float()
    return t

LR = 2e-5        # warm-started: lower LR than paper default
BETAS = (0.5, 0.999)
LAMBDA_L1 = 100.0
EPOCHS = 30

opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=BETAS)
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=BETAS)
bce = nn.BCEWithLogitsLoss()
l1  = nn.L1Loss()

def run_D(painting, edge, t_edge):
    return D(torch.cat([painting, edge, t_edge], dim=1))

def sample_preview(epoch, n=4):
    G.eval()
    with torch.no_grad():
        xs, ys, gs, ts = [], [], [], []
        for i, (x, y) in enumerate(val_dl):
            if i >= n: break
            x = x.to(DEVICE); y = y.to(DEVICE)
            g = G(x)
            t = trapped_ball_batch(g)
            xs.append(((x[0].cpu()+1)/2).permute(1,2,0).numpy())
            ys.append(((y[0,0].cpu()+1)/2).numpy())
            gs.append(((g[0,0].cpu()+1)/2).numpy())
            ts.append(((t[0,0].cpu()+1)/2).numpy())
        fig, axes = plt.subplots(len(xs), 4, figsize=(8, 2*len(xs)))
        for row, (xi, yi, gi, ti) in enumerate(zip(xs, ys, gs, ts)):
            axes[row,0].imshow(xi); axes[row,0].set_title('input')
            axes[row,1].imshow(yi, cmap='gray_r'); axes[row,1].set_title('target')
            axes[row,2].imshow(gi, cmap='gray_r'); axes[row,2].set_title('G(x)')
            axes[row,3].imshow(ti, cmap='gray_r'); axes[row,3].set_title('T(G(x))')
            for a in axes[row]: a.set_xticks([]); a.set_yticks([])
        fig.suptitle(f'epoch {epoch}'); fig.tight_layout()
        plt.show()
    G.train()

In [ ]:
history = {'d_loss': [], 'g_loss': [], 'g_l1': [], 'val_l1': []}

for epoch in range(1, EPOCHS+1):
    G.train(); D.train()
    d_ep, g_ep, l1_ep = 0.0, 0.0, 0.0
    for x, y in tqdm(train_dl, desc=f'epoch {epoch}/{EPOCHS}'):
        x = x.to(DEVICE); y = y.to(DEVICE)
        # === D step ===
        with torch.no_grad():
            g_out = G(x)
            t_fake = trapped_ball_batch(g_out)
            t_real = trapped_ball_batch(y)
        d_real = run_D(x, y, t_real)
        d_fake = run_D(x, g_out, t_fake)
        loss_D = 0.5 * (bce(d_real, torch.ones_like(d_real)) + bce(d_fake, torch.zeros_like(d_fake)))
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()
        # === G step ===
        g_out = G(x)
        with torch.no_grad():
            t_fake = trapped_ball_batch(g_out)
        d_fake2 = run_D(x, g_out, t_fake)
        loss_G_gan = bce(d_fake2, torch.ones_like(d_fake2))
        loss_G_l1  = l1(g_out, y) * LAMBDA_L1
        loss_G = loss_G_gan + loss_G_l1
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()
        d_ep += float(loss_D); g_ep += float(loss_G_gan); l1_ep += float(loss_G_l1)
    n = len(train_dl)
    # val L1
    G.eval()
    with torch.no_grad():
        vl = 0.0; vn = 0
        for x, y in val_dl:
            x=x.to(DEVICE); y=y.to(DEVICE)
            vl += float(l1(G(x), y)); vn += 1
    history['d_loss'].append(d_ep/n); history['g_loss'].append(g_ep/n)
    history['g_l1'].append(l1_ep/n); history['val_l1'].append(vl/max(vn,1))
    print(f'epoch {epoch}: D={d_ep/n:.3f} G_gan={g_ep/n:.3f} G_L1={l1_ep/n:.3f} val_L1={vl/max(vn,1):.3f}')
    if epoch % 3 == 0 or epoch == EPOCHS:
        sample_preview(epoch)

torch.save({'G': G.state_dict(), 'D': D.state_dict(), 'history': history}, CKPT)
print('saved', CKPT)

In [ ]:
# training curves
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(history['d_loss'], label='D'); ax[0].plot(history['g_loss'], label='G_gan'); ax[0].legend(); ax[0].set_title('adversarial losses')
ax[1].plot(history['g_l1'], label='G_L1 (train)'); ax[1].plot(history['val_l1'], label='L1 (val)'); ax[1].legend(); ax[1].set_title('L1 loss')
plt.show()